In [ ]:
#hailctl dataproc start nv-fusion \
#  --master-machine-type n1-highmem-16 \
#  --worker-machine-type n1-highmem-16 \
#  --master-boot-disk-size 450 \
#  --num-workers 0 \
#  --max-idle=100h \
#  --requester-pays-allow-all \
#  --big-executors \
#  --no-off-heap-memory \
#  --region us-central1 \
#  --network broad-allow
# You can try this, maybe you can make it a bigger machine (maybe n1-highmem-32) and decrease num of chunks 
# by mamking memory.chunks bigger
# it should take about half a day for each site you run if you pre-filter the plink files

In [1]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
udev             52G     0   52G   0% /dev
tmpfs            11G  2.2M   11G   1% /run
/dev/sda1       443G   18G  407G   5% /
tmpfs            52G     0   52G   0% /dev/shm
tmpfs           5.0M     0  5.0M   0% /run/lock
tmpfs           4.0M     0  4.0M   0% /sys/fs/cgroup
/dev/sda15      124M   12M  113M  10% /boot/efi
tmpfs            11G     0   11G   0% /run/user/106
tmpfs            11G     0   11G   0% /run/user/107
tmpfs            11G     0   11G   0% /run/user/108
tmpfs            11G     0   11G   0% /run/user/110
tmpfs            11G     0   11G   0% /run/user/111
tmpfs            11G     0   11G   0% /run/user/1017


In [2]:
# installing docker
%%bash
sudo apt-get update
sudo apt-get install -y docker.io

sudo systemctl enable docker
sudo systemctl start docker

docker --version

Get:1 file:/etc/apt/mirrors/debian.list Mirrorlist [30 B]
Get:5 file:/etc/apt/mirrors/debian-security.list Mirrorlist [39 B]
Get:7 https://storage.googleapis.com/goog-dataproc-bigtop-repo-us-central1/2_3_deb12_20251026_132200-RC01 dataproc InRelease [3708 B]
Get:8 https://download.docker.com/linux/debian bookworm InRelease [46.6 kB]
Hit:9 https://storage.googleapis.com/dataproc-bigtop-repo/2_3_deb12_20251026_132200-RC01 dataproc InRelease
Get:2 https://deb.debian.org/debian bookworm InRelease [151 kB]
Get:10 https://repo.mysql.com/apt/debian bookworm InRelease [30.0 kB]
Get:11 https://packages.cloud.google.com/apt google-compute-engine-bookworm-stable InRelease [1321 B]
Get:12 https://packages.cloud.google.com/apt cloud-sdk-bookworm InRelease [1656 B]
Get:3 https://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:4 https://deb.debian.org/debian bookworm-backports InRelease [59.4 kB]
Get:6 https://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:13 

dpkg-preconfigure: unable to re-open stdin: No such file or directory


Fetched 68.3 MB in 1s (79.9 MB/s)
Selecting previously unselected package runc.
(Reading database ... 169280 files and directories currently installed.)
Preparing to unpack .../00-runc_1.1.5+ds1-1+deb12u1_amd64.deb ...
Unpacking runc (1.1.5+ds1-1+deb12u1) ...
Selecting previously unselected package containerd.
Preparing to unpack .../01-containerd_1.6.20~ds1-1+deb12u3_amd64.deb ...
Unpacking containerd (1.6.20~ds1-1+deb12u3) ...
Selecting previously unselected package tini.
Preparing to unpack .../02-tini_0.19.0-1+b3_amd64.deb ...
Unpacking tini (0.19.0-1+b3) ...
Selecting previously unselected package docker.io.
Preparing to unpack .../03-docker.io_20.10.24+dfsg1-1+deb12u1+b6_amd64.deb ...
Unpacking docker.io (20.10.24+dfsg1-1+deb12u1+b6) ...
Selecting previously unselected package cgroupfs-mount.
Preparing to unpack .../04-cgroupfs-mount_1.4_all.deb ...
Unpacking cgroupfs-mount (1.4) ...
Selecting previously unselected package libprotobuf32:amd64.
Preparing to unpack .../05-libprotob

Synchronizing state of docker.service with SysV service script with /lib/systemd/systemd-sysv-install.
Executing: /lib/systemd/systemd-sysv-install enable docker


Docker version 20.10.24+dfsg1, build 297e128


In [3]:
# login to docker
%%bash
sudo usermod -aG docker $USER

In [4]:
# docker test
!sudo docker run hello-world

Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world

Digest: sha256:0e760fdfbc48ba8041e7c6db999bb40bfca508b4be580ac75d32c4e29d202ce1
Status: Downloaded newer image for hello-world:latest

Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https

In [5]:
# pull POLMM docker image
!sudo docker pull nsvalencia/polmm:v6

v6: Pulling from nsvalencia/polmm

a40fdbce: Pulling fs layer 
16466c28: Pulling fs layer 
4fff2683: Pulling fs layer 
522e412e: Pulling fs layer 
ae7e0f4a: Pulling fs layer 
6eeac82e: Pulling fs layer 
c5adf2df: Pulling fs layer 
9948e974: Pulling fs layer 
e76b4431: Pulling fs layer 
6fdc4f85: Pulling fs layer 
Digest: sha256:dff05ad0f833c2d524611dbb1320c41e6d1ed6da786a0bf431cac725a399896d
Status: Downloaded newer image for nsvalencia/polmm:v6
docker.io/nsvalencia/polmm:v6


In [6]:
!sudo docker run --rm nsvalencia/polmm:v6 Rscript -e "library(POLMM); cat('POLMM loaded\n')"

POLMM loaded


In [7]:
!mkdir plink_files
!mkdir null_models

In [23]:
!cd /home/hail/plink_files

In [9]:
# Download all Plink files being used for association testing
# Note difference with Khat Use, we are using combined plink files, not per-chr ones
%%bash

gsutil -m cp \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/AAU_passed_all_qc.bed \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/AAU_passed_all_qc.bim \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/AAU_passed_all_qc.fam \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/KEMRI_passed_all_qc.bed \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/KEMRI_passed_all_qc.bim \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/KEMRI_passed_all_qc.fam \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Moi_passed_all_qc.bed \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Moi_passed_all_qc.bim \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Moi_passed_all_qc.fam \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Uganda_passed_all_qc.bed \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Uganda_passed_all_qc.bim \
  gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Uganda_passed_all_qc.fam \
  .

Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/AAU_passed_all_qc.bed...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/AAU_passed_all_qc.bim...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/AAU_passed_all_qc.fam...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/KEMRI_passed_all_qc.bed...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Moi_passed_all_qc.fam...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/KEMRI_passed_all_qc.bim...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Uganda_passed_all_qc.fam...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/KEMRI_passed_all_qc.fam...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Moi_passed_all_qc.bed...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/Moi_passed_all_qc.bim...
Copying gs://neurogap-bge-imputed-regional/lerato/wave2/plink

In [ ]:
# Following this, we pre-filter the data to MAF 0.01 (minMAF 1e-2) 
# Without doing this, it takes a lot longer to compute because POLMM writes out pretty slowly
# This should cut compute times for all sites roughly equal to the percentage of vars that were dropped

In [10]:

%%bash
mkdir -p /home/hail/bin
cd /home/hail/bin

wget -q https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_x86_64_20251123.zip
unzip -o plink2_linux_x86_64_20251123.zip
chmod +x plink2

/home/hail/bin/plink2 --version

Archive:  plink2_linux_x86_64_20251123.zip
  inflating: plink2                  
  inflating: vcf_subset              
  inflating: intel-simplified-software-license.txt  
PLINK v2.0.0-a.6.28LM 64-bit Intel (23 Nov 2025)


In [25]:
%%bash
cat > /home/hail/plink_files/run_maf_filter.sh << 'EOF'
#!/bin/bash

cd /home/hail/plink_files

run_site () {
    site=$1

    echo "Starting ${site} at $(date)"

    /home/hail/bin/plink2 \
        --threads 4 \
        --bfile ${site}_passed_all_qc \
        --maf 0.01 \
        --make-bed \
        --out ${site}_passed_all_qc_maf01 \
        > ${site}_maf01.log 2>&1

    echo "Finished ${site} at $(date)"
}

run_site AAU &
PID1=$!

run_site KEMRI &
PID2=$!

run_site Moi &
PID3=$!

run_site Uganda &
PID4=$!

echo "PIDs: $PID1 $PID2 $PID3 $PID4"

wait $PID1
wait $PID2
wait $PID3
wait $PID4

echo "ALL DONE $(date)"
EOF

chmod +x /home/hail/plink_files/run_maf_filter.sh

In [26]:
%%bash
cd /home/hail/plink_files

nohup ./run_maf_filter.sh > maf_filter_master.log 2>&1 &
echo $! > maf_filter.pid

echo "Master PID: $(cat maf_filter.pid)"

Master PID: 28939


In [29]:
%%bash
echo "PID file:"
cat /home/hail/plink_files/maf_filter.pid || true

echo
echo "Master log:"
cat /home/hail/plink_files/maf_filter_master.log || true

echo
echo "Files:"
ls -lh /home/hail/plink_files | grep -E "maf01|maf_filter|plink2" || true

PID file:
28939

Master log:
PIDs: 28941 28942 28943 28946
Starting KEMRI at Tue Jun  2 01:14:51 UTC 2026
Starting AAU at Tue Jun  2 01:14:51 UTC 2026
Starting Moi at Tue Jun  2 01:14:51 UTC 2026
Starting Uganda at Tue Jun  2 01:14:51 UTC 2026

Files:
-rw-r--r-- 1 root root    0 Jun  2 01:14 AAU_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 AAU_passed_all_qc_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 KEMRI_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 KEMRI_passed_all_qc_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 Moi_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 Moi_passed_all_qc_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 Uganda_maf01.log
-rw-r--r-- 1 root root    0 Jun  2 01:14 Uganda_passed_all_qc_maf01.log
-rw-r--r-- 1 root root    6 Jun  2 01:14 maf_filter.pid
-rw-r--r-- 1 root root  215 Jun  2 01:14 maf_filter_master.log
-rwxr-xr-x 1 root root  567 Jun  2 01:14 run_maf_filter.sh


In [67]:
!wc -l /home/plink_files/*maf01.bim
# number of vars expected in output from association testing as well

wc: '/home/plink_files/*maf01.bim': No such file or directory


In [41]:
%%bash
for site in AAU KEMRI Moi Uganda
do
  echo "----- $site -----"
  tail -5 /home/hail/plink_files/${site}_maf01.log
done

----- AAU -----
10980551 variants remaining after main filters.
Writing AAU_passed_all_qc_maf01.fam ... done.
Writing AAU_passed_all_qc_maf01.bim ... done.
Writing AAU_passed_all_qc_maf01.bed ... 1011121314151617181920212223242526272829303132333435363738394041424344454647484950515253545557585960616263646566676869707172737475767778798081828384858687888990919293949596979899done.
End time: Tue Jun  2 03:42:04 2026
----- KEMRI -----
13876676 variants remaining after main filters.
Writing KEMRI_passed_all_qc_maf01.fam ... done.
Writing KEMRI_passed_all_qc_maf01.bim ... done.
Writing KEMRI_passed_all_qc_maf01.bed ... 1011121314151617181920212223242526272829303132333435363738394041424344454647484950515354555657585960616263646566676869707172737475767778798081828384858687888990919293949596979899done.
End time: Tue Jun  2 02:10:10 2026
----- Moi -----
13452287 variants remaining after main filters.
Writing Moi_passed_all_qc_maf01.fam ... done.
Writing Moi_passed_all_qc_maf01.bim ... done.
Writin

In [50]:
# Viewing outputted plink files
%cd /home/hail
!ls
%cd /home/hail/plink_files
!ls

/home/hail
bin			       plink_files
hail-0.2.137-py3-none-any.whl  sparkmonitor-0.0.12-py3-none-any.whl
null_models		       sparkmonitor_kernelextension.log
/home/hail/plink_files
AAU_maf01.log		       Moi_passed_all_qc.bim
AAU_passed_all_qc.bed	       Moi_passed_all_qc.fam
AAU_passed_all_qc.bim	       Moi_passed_all_qc_maf01.bed
AAU_passed_all_qc.fam	       Moi_passed_all_qc_maf01.bim
AAU_passed_all_qc_maf01.bed    Moi_passed_all_qc_maf01.fam
AAU_passed_all_qc_maf01.bim    Moi_passed_all_qc_maf01.log
AAU_passed_all_qc_maf01.fam    Uganda_maf01.log
AAU_passed_all_qc_maf01.log    Uganda_passed_all_qc.bed
KEMRI_maf01.log		       Uganda_passed_all_qc.bim
KEMRI_passed_all_qc.bed        Uganda_passed_all_qc.fam
KEMRI_passed_all_qc.bim        Uganda_passed_all_qc_maf01.bed
KEMRI_passed_all_qc.fam        Uganda_passed_all_qc_maf01.bim
KEMRI_passed_all_qc_maf01.bed  Uganda_passed_all_qc_maf01.fam
KEMRI_passed_all_qc_maf01.bim  Uganda_passed_all_qc_maf01.log
KEMRI_passed_all_qc_maf01.fam  maf_f

In [39]:
# Downloading null models
%cd /home/hail/null_models
!gsutil -m cp "gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_null/studySite_numeric_covars/*" .

/home/hail/null_models
Copying gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_null/studySite_numeric_covars/gp08_AAU_assist_khat_amt_polmm_null.rds...
Copying gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_null/studySite_numeric_covars/gp08_KEMRI_assist_khat_amt_polmm_null.rds...
Copying gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_null/studySite_numeric_covars/gp08_Moi_assist_khat_amt_polmm_null.rds...
Copying gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_null/studySite_numeric_covars/gp08_Uganda_assist_khat_amt_polmm_null.rds...
\ [4/4 files][ 48.6 MiB/ 48.6 MiB] 100% Done                                    
Operation completed over 4 objects/48.6 MiB.                                     


In [51]:
# Copying these files into google cloud for backup
!gsutil cp /home/hail/plink_files/*maf01* gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/minmaf01

Copying file:///home/hail/plink_files/AAU_maf01.log [Content-Type=application/octet-stream]...
Copying file:///home/hail/plink_files/AAU_passed_all_qc_maf01.bed [Content-Type=application/vnd.realvnc.bed]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

Copying file:///home/hail/plink_files/AAU_passed_all_qc_maf01.bim [Content-Type=application/octet-stream]..

In [52]:
!ls -lh /home/hail/null_models

total 49M
-rw-r--r-- 1 root root  20M Jun  2 03:29 gp08_AAU_assist_khat_amt_polmm_null.rds
-rw-r--r-- 1 root root 5.0M Jun  2 03:29 gp08_KEMRI_assist_khat_amt_polmm_null.rds
-rw-r--r-- 1 root root 7.0M Jun  2 03:29 gp08_Moi_assist_khat_amt_polmm_null.rds
-rw-r--r-- 1 root root  18M Jun  2 03:29 gp08_Uganda_assist_khat_amt_polmm_null.rds


In [54]:
!mkdir -p /home/hail/polmm/assocTest_outputs
!mkdir -p /home/hail/polmm/logs_assocTest

In [ ]:
# What is below is the primary set of arguments from POLMM
# PlinkFile = plink_prefix = the maf01 files we made
# memory.chunk = how many chunks the job is split up into, equation comes out to (some big number)memory.chunk
# minMAF = 1e2 --> left this, doesn't functionally do anything considering that we already filtered the data
# maxMissing = variant-level missingness filter

In [55]:
%%writefile /home/hail/polmm/run_polmm_assoc.R
args <- commandArgs(trailingOnly = TRUE)

options(error = function() {
  cat("\n===== R TRACEBACK =====\n")
  traceback(2)
  cat("=======================\n")
  quit(status = 1)
})

null_model_file <- args[1]
plink_prefix <- args[2]
output_prefix <- args[3]

library(POLMM)

cat("Null model:", null_model_file, "\n")
cat("PLINK prefix:", plink_prefix, "\n")
cat("Output prefix:", output_prefix, "\n")

objNull <- readRDS(null_model_file)
chrVec <- names(objNull$LOCOList)

cat("Chromosomes from LOCOList:", paste(chrVec, collapse = ", "), "\n")

POLMM.plink(
  objNull       = objNull,
  PlinkFile     = plink_prefix,
  output.file   = output_prefix,
  chrVec.plink  = chrVec,
  memory.chunk  = 1.5,
  SPAcutoff     = 2,
  minMAF        = 1e-2,
  maxMissing    = 0.15,
  impute.method = "fixed",
  G.model       = "Add"
)

cat("POLMM association test complete\n")

Writing /home/hail/polmm/run_polmm_assoc.R


In [ ]:
# What is below is the dispatcher that will run POLMM association testing for each site
# Note the no-UCT presence

# Below this dispatch there are a bunch of commands left that you can run to view progress or see errors

In [56]:
%%bash
cat > /home/hail/polmm/run_all_polmm_assoc.sh <<'EOF'
#!/usr/bin/env bash
set -uo pipefail

WORKDIR="/home/hail/polmm"
NULL_DIR="/home/hail/null_models"
PLINK_DIR="/home/hail/plink_files"
OUTPUT_DIR="${WORKDIR}/assocTest_outputs"
LOGDIR="${WORKDIR}/logs_assocTest"
GCS_OUT="gs://neurogap-bge-imputed-regional/nico/khat_gwas/polmm_assoc/studysite_covar_all"

DOCKER_IMAGE="nsvalencia/polmm:v6"
R_SCRIPT="run_polmm_assoc.R"

SITES=("AAU" "KEMRI" "Moi" "Uganda")

mkdir -p "${OUTPUT_DIR}" "${LOGDIR}"
cd "${WORKDIR}"

for SITE in "${SITES[@]}"; do
    echo "=================================================="
    echo "Starting POLMM association test for ${SITE}"
    echo "=================================================="

    PLINK_PREFIX="/plink_files/${SITE}_passed_all_qc_maf01"
    NULL_MODEL="/null_models/gp08_${SITE}_assist_khat_amt_polmm_null.rds"
    OUT_PREFIX="assoc_${SITE}_studysite_covar_all"
    LOCAL_OUT_PREFIX="assocTest_outputs/${OUT_PREFIX}"

    LOG="${LOGDIR}/${SITE}_assocTest_$(date -u +%Y%m%dT%H%M%SZ).log"

    sudo docker run --rm \
        -v "${WORKDIR}:/data" \
        -v "${NULL_DIR}:/null_models" \
        -v "${PLINK_DIR}:/plink_files" \
        -w /data \
        "${DOCKER_IMAGE}" \
        Rscript "${R_SCRIPT}" \
            "${NULL_MODEL}" \
            "${PLINK_PREFIX}" \
            "${LOCAL_OUT_PREFIX}" \
        > "${LOG}" 2>&1

    RC=$?

    if [[ ${RC} -eq 0 ]]; then
        echo "${SITE} completed successfully. Uploading outputs..."
        gsutil -m cp "${OUTPUT_DIR}/${OUT_PREFIX}"* "${GCS_OUT}/"
    else
        echo "${SITE} failed with exit code ${RC}. See log: ${LOG}"
        exit ${RC}
    fi
done
EOF

chmod +x /home/hail/polmm/run_all_polmm_assoc.sh

In [57]:
%%bash
cd /home/hail/polmm

nohup bash run_all_polmm_assoc.sh > run_all_polmm_assoc.nohup.log 2>&1 &

echo $! > run_all_polmm_assoc.pid
echo "Started POLMM assoc loop with PID:"
cat run_all_polmm_assoc.pid

Started POLMM assoc loop with PID:
98042


In [58]:
!cat /home/hail/polmm/run_all_polmm_assoc.pid
!ps -fp $(cat /home/hail/polmm/run_all_polmm_assoc.pid)

98042
UID          PID    PPID  C STIME TTY          TIME CMD
root       98042       1  0 04:32 ?        00:00:00 bash run_all_polmm_assoc.sh


In [61]:
!tail -50 /home/hail/polmm/run_all_polmm_assoc.nohup.log

Starting POLMM association test for AAU
AAU completed successfully. Uploading outputs...
Copying file:///home/hail/polmm/assocTest_outputs/assoc_AAU_studysite_covar_all [Content-Type=application/octet-stream]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.


Operation completed over 1 objects/1.9 GiB.                                      
Starting POLMM association te

In [62]:
!ls -ltr /home/hail/polmm/logs_assocTest

total 192
-rw-r--r-- 1 root root 142905 Jun  2 13:13 AAU_assocTest_20260602T043244Z.log
-rw-r--r-- 1 root root  44482 Jun  2 16:38 KEMRI_assocTest_20260602T131347Z.log


In [64]:
!cat /home/hail/polmm/logs_assocTest/AAU_assocTest_20260602T043244Z.log

Null model: /null_models/gp08_AAU_assist_khat_amt_polmm_null.rds 
PLINK prefix: /plink_files/AAU_passed_all_qc_maf01 
Output prefix: assocTest_outputs/assoc_AAU_studysite_covar_all 
Chromosomes from LOCOList: 2, 4, 7, 21, 11, 15, 20, 3, 8, 19, 12, 14, 18, 16, 10, 17, 5, 1, 6, 9, 22, 13 
[1] "Totally 10980551 markers in plink files."
[1] "After filtering by 'chrVec.plink', 10980551 markers are left."
[1] "Split all markers into 297 chunks."
[1] "Each chunk includes less than 36974 markers."
[1] "Analyzing chunk 1/297."
start
file = /plink_files/AAU_passed_all_qc_maf01
Tue Jun  2 04:34:21 2026 - read bim
Tue Jun  2 04:34:27 2026 - read fam
extract 36974 marker and 10142 sample out of 10980551 marker and 10142 sample
Tue Jun  2 04:34:30 2026 - read bed
Binary PLINK BED file is ready to read
build a look-up table
assigned 374990308 values 
allocate dim and dimnames
Tue Jun  2 04:34:33 2026 - end
Totally 36974 markers to analyze!!
Start analyzing chromosome 1 
[1] "Analyzing chunk 2/297."
s

In [26]:
!sudo docker ps
!cat /home/hail/polmm/run_all_polmm_assoc.nohup.log

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES
Starting POLMM association test for AAU
AAU failed with exit code 1. See log: /home/hail/polmm/logs_assocTest/AAU_assocTest_20260601T012253Z.log


In [28]:
# Job running check, change /home/hail/polmm/logs_assocTest/*log to whatever the log above
# might be if it fails
!grep -i -A5 -B5 "error" /home/hail/polmm/logs_assocTest/AAU_assocTest_20260601T012253Z.log

In [29]:
# Job running check
!grep -i "non-numeric matrix extent" /home/hail/polmm/logs_assocTest/AAU_assocTest_20260601T012253Z.log


In [30]:
!dmesg -T | tail -80

[Sun May 31 23:02:48 2026] systemd[1]: Starting modprobe@loop.service - Load Kernel Module loop...
[Sun May 31 23:02:48 2026] loop: module loaded
[Sun May 31 23:02:48 2026] systemd[1]: netplan-ovs-cleanup.service - OpenVSwitch configuration for cleanup was skipped because of an unmet condition check (ConditionFileIsExecutable=/usr/bin/ovs-vsctl).
[Sun May 31 23:02:48 2026] systemd[1]: Starting systemd-fsck-root.service - File System Check on Root Device...
[Sun May 31 23:02:48 2026] systemd[1]: Starting systemd-journald.service - Journal Service...
[Sun May 31 23:02:48 2026] systemd[1]: Starting systemd-modules-load.service - Load Kernel Modules...
[Sun May 31 23:02:48 2026] systemd[1]: Starting systemd-network-generator.service - Generate network units from Kernel command line...
[Sun May 31 23:02:48 2026] systemd[1]: Starting systemd-udev-trigger.service - Coldplug All udev Devices...
[Sun May 31 23:02:48 2026] systemd[1]: Started systemd-journald.service - Journal Service.
[Sun May 

In [31]:
!grep -i "error" /home/hail/polmm/logs_assocTest/AAU_assocTest_20260601T012253Z.log
!grep -i "warning" /home/hail/polmm/logs_assocTest/AAU_assocTest_20260601T012253Z.log
!grep -i "non-numeric" /home/hail/polmm/logs_assocTest/AAU_assocTest_20260601T012253Z.log

In [32]:
# Viewing contents of output directory
!ls -lh /home/hail/polmm/assocTest_outputs

total 63M
-rw-r--r-- 1 root root 63M Jun  1 01:58 assoc_AAU_studysite_covar_all


In [68]:
# Sanity check to make sure files look sensible
# Note, POLMM doesn't annotate on BPs, will do that later
!head /home/hail/polmm/assocTest_outputs/*
!ls /home/hail/polmm/assocTest_outputs/
!wc -l /home/hail/polmm/assocTest_outputs/*

==> /home/hail/polmm/assocTest_outputs/assoc_AAU_studysite_covar_all <==
SNPID	chr	MAF	missing.rate	Stat	VarW	VarP	beta	pval.norm	pval.spa	switch.allele
"rs868589190"	"1"	"0.0467921129016114"	"0.0148885821337014"	"-2.3710647134097"	"192.048874850398"	"168.704799065689"	"-0.0140545184638551"	"0.855151827802668"	"0.855151827802668"	"TRUE"
"rs779143624"	"1"	"0.0292070217917676"	"0.0226779727864327"	"2.6044530909992"	"121.83991961352"	"107.029937939446"	"0.024333874625553"	"0.801236805570581"	"0.801236805570581"	"TRUE"
"rs201833382"	"1"	"0.0273087876322213"	"0.030565963320844"	"-3.48786694586593"	"114.326660206008"	"100.429936144798"	"-0.0347293554069095"	"0.727810651484239"	"0.727810651484239"	"TRUE"
"rs1461261258"	"1"	"0.019170301247591"	"0.0279037665154802"	"-6.42346126980518"	"85.120249355216"	"74.7736459017495"	"-0.0859054175082679"	"0.457579021025923"	"0.457579021025923"	"TRUE"
"rs200784459"	"1"	"0.0125523012552301"	"0.0102543876947348"	"20.4371005107041"	"55.8426802364288"	"49.05484

In [34]:
# indicate where you'd like to output the files
!gsutil cp /home/hail/polmm/assocTest_outputs/* gsneurogap-bge-imputed-regional/nico/khat_gwas/polmm_assoc/studysite_covar_all